# NOVA 2: Processed Culinary Ingredients

### What are Processed Culinary Ingredients?
Substances derived from NOVA 1 foods or from nature through physical and industrial processes such as pressing, refining, grinding, milling, and drying. Examples include vegetable oils, butter, lard, sugar, molasses, honey, vinegars, and salt. They are rarely consumed alone; instead, they are used to season, cook, and prepare freshly made dishes.

In this project, a product remains in NOVA 2 only if every ingredient is one of these core substances. Any product containing additional ingredients — including preservatives, anti-caking agents, emulsifiers, flavors, colors, or sweeteners — was conservatively removed from NOVA 2, as such products no longer represent a single culinary ingredient

*Source: Monteiro et al., 2018; FAO, 2019 (NOVA classification framework)*


In [1]:
# NOVA 2 allowed substances (from Monteiro definition: pressed/refined/ground/milled/dried)
nova2_allowed = [
    'OIL', 'VEGETABLE OIL', 'OLIVE OIL', 'BUTTER', 'LARD', 'GHEE', 'TALLOW', 'COCONUT FAT',
    'SUGAR', 'WHITE SUGAR', 'BROWN SUGAR', 'MOLASSES', 'HONEY', 'MAPLE SYRUP', 
    'SALT', 'SEA SALT', 'IODIZED SALT',
    'VINEGAR', 'APPLE CIDER VINEGAR', 'BALSAMIC VINEGAR',
    'STARCH', 'CORNSTARCH', 'POTATO STARCH'
]


# The goal is to remove products wrongly assigned to NOVA 2 by keyword classification. A true 
# NOVA 2 product is a single substance (oil, sugar, salt, vinegar, native starch) 
# with no additives. (Source: Monteiro et al., NOVA framework).
# Will isolate NOVA 2 from the classified dataset (48,240 products).


In [2]:
import pandas as pd

df = pd.read_csv('branded_food_cleaned_classified.csv')

print("NOVA distribution: ")
print(df["nova_category"].value_counts())

NOVA distribution: 
nova_category
NOVA 4 - Ultra-Processed                    1076861
Other/Unclassified                           436055
NOVA 3 - Processed                           325482
NOVA 1 - Unprocessed/Minimally Processed      77038
NOVA 2 - Culinary Ingredient                  48240
Name: count, dtype: int64


In [3]:
# Isolate NOVA 2 from the full dataset
nova2 = df[df['nova_category'] == 'NOVA 2 - Culinary Ingredient'].copy()
print(f"NOVA 2 total: {len(nova2):,}")
print(f"\nNull check:")
print(nova2.isnull().sum())

NOVA 2 total: 48,240

Null check:
fdc_id                      0
brand_owner               293
branded_food_category       0
ingredients              1025
market_country              0
nova_category               0
dtype: int64


In [4]:
# Build a single regex pattern from the allowed list
allowed_pattern = '|'.join(nova2_allowed)

def is_true_nova2(ingredient_string):
    """Returns True only if EVERY ingredient matches an allowed NOVA 2 substance."""
    if pd.isnull(ingredient_string):
        return None  # handled separately (self-evident nulls)
    # Split into individual ingredients
    parts = [p.strip().upper() for p in ingredient_string.split(',')]
    parts = [p for p in parts if p]  # drop empties
    # Every part must contain at least one allowed substance
    for part in parts:
        if not any(allowed in part for allowed in nova2_allowed):
            return False
    return True

# Apply to the validatable rows (non-null ingredients)
nova2_with_ingredients = nova2[nova2['ingredients'].notna()].copy()
nova2_with_ingredients['is_nova2'] = nova2_with_ingredients['ingredients'].apply(is_true_nova2)

# Results
true_nova2 = nova2_with_ingredients[nova2_with_ingredients['is_nova2'] == True]
impostors = nova2_with_ingredients[nova2_with_ingredients['is_nova2'] == False]

print(f"Validatable NOVA 2 rows: {len(nova2_with_ingredients):,}")
print(f"  True NOVA 2 (all ingredients allowed): {len(true_nova2):,}")
print(f"  Impostors (contain non-NOVA-2 ingredients): {len(impostors):,}")

Validatable NOVA 2 rows: 47,215
  True NOVA 2 (all ingredients allowed): 24,070
  Impostors (contain non-NOVA-2 ingredients): 23,145


In [5]:
# Profile what ingredients make products impostors
from collections import Counter

impostor_ingredients = []
for item in impostors['ingredients'].dropna():
    parts = [ing.strip().upper() for ing in item.split(',')]
    parts = [p for p in parts if p]
    impostor_ingredients.extend(parts)

impostor_counts = pd.DataFrame(Counter(impostor_ingredients).most_common(30),
                               columns=['ingredient', 'count'])
impostor_counts['% of impostors'] = (impostor_counts['count'] / len(impostors) * 100).round(2)

print(f"Total impostor products: {len(impostors):,}")
print(f"Unique impostor ingredients: {len(set(impostor_ingredients)):,}\n")
impostor_counts

Total impostor products: 23,145
Unique impostor ingredients: 5,431



,ingredient,count,% of impostors
0,WATER,5506,23.79
1,SALT,3489,15.07
2,HIGH FRUCTOSE CORN SYRUP,2655,11.47
3,CORN SYRUP,2525,10.91
4,RIBOFLAVIN,2351,10.16
5,NIACIN,2192,9.47
6,CARAMEL COLOR,2161,9.34
7,CITRIC ACID,1961,8.47
8,SUGAR,1810,7.82
9,REDUCED IRON,1761,7.61


In [6]:
# Final clean NOVA 2
nova2_clean = pd.concat([
    true_nova2,
    nova2[nova2['ingredients'].isnull()]   # the 1,025 self-evident NOVA 2
])

print(f"Final clean NOVA 2: {len(nova2_clean):,}")
print(f"  Validated survivors: {len(true_nova2):,}")
print(f"  Self-evident (null ingredients): {len(nova2[nova2['ingredients'].isnull()]):,}")
print(f"\nDropped impostors: {len(impostors):,}")
print(f"Original NOVA 2: {len(nova2):,}")

Final clean NOVA 2: 25,095
  Validated survivors: 24,070
  Self-evident (null ingredients): 1,025

Dropped impostors: 23,145
Original NOVA 2: 48,240


In [7]:
# Checking the NOVA 2 ingredients after dropping the imposter
# Profile the ingredients in the validated NOVA 2 set
clean_ingredients = []
for item in nova2_clean['ingredients'].dropna():
    parts = [ing.strip().upper().rstrip('.').strip() for ing in item.split(',')]
    parts = [p for p in parts if p]
    clean_ingredients.extend(parts)

clean_counts = pd.DataFrame(Counter(clean_ingredients).most_common(30),
                            columns=['ingredient', 'count'])
clean_counts['% of clean NOVA 2'] = (clean_counts['count'] / len(nova2_clean) * 100).round(2)

print(f"Clean NOVA 2 products: {len(nova2_clean):,}")
print(f"(excludes {len(nova2[nova2['ingredients'].isnull()]):,} null-ingredient rows from this ingredient count)")
print(f"Unique ingredients: {len(set(clean_ingredients)):,}\n")
clean_counts

Clean NOVA 2 products: 25,095
(excludes 1,025 null-ingredient rows from this ingredient count)
Unique ingredients: 1,269



,ingredient,count,% of clean NOVA 2
0,EXTRA VIRGIN OLIVE OIL,2583,10.29
1,HONEY,1837,7.32
2,SUGAR,1819,7.25
3,SOYBEAN OIL,1354,5.40
4,CANOLA OIL,1239,4.94
5,ORGANIC EXTRA VIRGIN OLIVE OIL,840,3.35
6,CORN OIL,594,2.37
7,OLIVE OIL,420,1.67
8,CORNSTARCH,394,1.57
9,ORGANIC HONEY,394,1.57


In [8]:
# Saved the ingredient frequency
clean_counts.to_csv('nova2_ingredient_frequency.csv', index=False)
print("Saved: nova2_ingredient_frequency.csv")

Saved: nova2_ingredient_frequency.csv


In [9]:
# Saved clean NOVA 2 product set
nova2_clean.to_csv('nova2_clean_validated.csv', index=False)
print(f"Saved: nova2_clean_validated.csv ({len(nova2_clean):,} rows)")

Saved: nova2_clean_validated.csv (25,095 rows)


## Results
- Original NOVA 2: **48,240**
- Impostors removed: **23,145 (49% of validatable products)**
- **Final validated NOVA 2: 25,095**
## Conclusion
Nearly half of keyword-classified NOVA 2 products did not meet the definition — 
the contamination was almost entirely NOVA 3/4 (syrups, enriched flours, additives). 
Category labels and keyword matching alone are unreliable for NOVA classification; 
ingredient-level validation is required.
## Limitation
~87 products (0.18%) with parenthetical ingredient breakdowns were conservatively 
flagged as impostors and not recovered.